## 装饰器
### 装饰器的关键特征：
### 接受一个函数作为参数
### 返回一个可调用对象（通常是函数）
### 可以使用@语法

In [2]:
import torch

'''
基本语法
def decorator(func):
    def wrapper(*args, **kwargs):
        # 在调用原函数前的操作
        result = func(*args, **kwargs)
        # 在调用原函数后的操作
        return result
    return wrapper

@decorator
def my_function():
    pass
'''

'\n基本语法\ndef decorator(func):\n    def wrapper(*args, **kwargs):\n        # 在调用原函数前的操作\n        result = func(*args, **kwargs)\n        # 在调用原函数后的操作\n        return result\n    return wrapper\n\n@decorator\ndef my_function():\n    pass\n'

In [3]:
'''
题目1：基础装饰器实现
功能：记录函数执行时间并打印
要求：
1. 装饰器应该打印函数名和执行时间
2. 保持原函数的元信息（可以使用functools.wraps）
3. 支持带参数和返回值的函数
'''

'\n题目1：基础装饰器实现\n功能：记录函数执行时间并打印\n要求：\n1. 装饰器应该打印函数名和执行时间\n2. 保持原函数的元信息（可以使用functools.wraps）\n3. 支持带参数和返回值的函数\n'

In [4]:
import time
from functools import wraps
def time_decorator(func):
    @wraps(func)  # 保持原函数的元信息
    def wrapper(*args, **kwargs):
        # 记录开始时间
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        duration = end_time - start_time
        print(f"[{func.__name__}] 执行时间: {duration:.4f}秒")
        return result
    return wrapper


In [5]:
@time_decorator
def train_model(epochs=10):
    """模拟模型训练"""
    time.sleep(0.1 * epochs)
    return f"训练完成 {epochs} 个epoch"
train_model()

[train_model] 执行时间: 1.0001秒


'训练完成 10 个epoch'

In [6]:
'''
题目2：带参数的装饰器
功能：当函数执行失败时自动重试
参数：
- max_retries: 最大重试次数
- delay: 每次重试之间的延迟（秒）

要求：
1. 装饰器本身接受参数
2. 只在发生特定异常时重试（如ConnectionError）
3. 重试之间有延迟
4. 达到最大重试次数后抛出异常
'''
def retry_decorator(max_retries=3, delay=1.0):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(max_retries):
                try:
                    result = func(*args, **kwargs)
                    return result
                except ConnectionError as e:
                    if i < max_retries-1:
                        print(f"函数[{func.__name__}]执行失败，重试第{i}次,错误信息：{e}")
                        time.sleep(delay)
                    else:
                        print(f"函数[{func.__name__}]执行失败，已达到最大重试次数，放弃重试,错误信息：{e}")
        return wrapper
    return decorator


# 测试用例
@retry_decorator(max_retries=3, delay=0.5)
def fetch_data_from_api(url):
    """模拟从API获取数据（可能失败）"""
    import random
    if random.random() < 0.7:  # 70%的概率失败
        raise ConnectionError("API连接失败")
    return {"data": "成功获取的数据"}

fetch_data_from_api('ddd')

函数[fetch_data_from_api]执行失败，重试第0次,错误信息：API连接失败
函数[fetch_data_from_api]执行失败，重试第1次,错误信息：API连接失败


{'data': '成功获取的数据'}

In [7]:
'''
题目3：类装饰器
功能：统计函数的调用次数和最近调用时间
要求：
1. 使用类实现装饰器
2. 统计每个被装饰函数的调用次数
3. 记录最近一次调用的时间
4. 可以通过属性访问统计信息
'''
from datetime import datetime
import torch
class CallCounter:
    def __init__(self, func):
        wraps(func)(self)
        self.func = func
        self.call_count = 0
        self.last_called_time = None


    def __call__(self, *args, **kwargs):
        self.call_count += 1
        self.last_called_time = datetime.now()
        return self.func(*args, **kwargs)
    def get_stats(self):
        """获取统计信息"""
        return {
            'function_name': self.func.__name__,
            'call_count': self.call_count,
            'last_called': self.last_called_time.strftime('%Y-%m-%d %H:%M:%S')
                          if self.last_called_time else '从未调用'
        }

    def reset_stats(self):
        """重置统计信息"""
        self.call_count = 0
        self.last_called_time = None


# 测试用例
@CallCounter
def forward_pass(input_tensor):
    """模拟前向传播"""
    time.sleep(0.01)
    return input_tensor * 2

forward_pass(torch.tensor([1,2,3]))
print(f"统计: {forward_pass.get_stats()}")

统计: {'function_name': 'forward_pass', 'call_count': 1, 'last_called': '2026-01-08 10:23:41'}


In [8]:
'''
@符号的限制和陷阱
1. 顺序问题:执行顺序是从下往上
'''
def decorator1(func):
    print("decorator1")
    return func

def decorator2(func):
    print("decorator2")
    return func

# 顺序很重要
@decorator1
@decorator2
def func1():
    pass
# 输出：decorator2 → decorator1

@decorator2
@decorator1
def func2():
    pass
# 输出：decorator1 → decorator2

decorator2
decorator1
decorator1
decorator2


In [9]:
'''
2. 原信息丢失，需要用wraps解决
'''
def bad_decorator(func):
    def wrapper():
        return func()
    return wrapper

def good_decorator(func):
    @wraps(func)  # 保持元信息
    def wrapper():
        return func()
    return wrapper

@bad_decorator
def func1():
    """Function 1"""
    pass

@good_decorator
def func2():
    """Function 2"""
    pass

print(func1.__name__)  # wrapper ❌
print(func1.__doc__)   # None ❌
print(func2.__name__)  # func2 ✅
print(func2.__doc__)   # Function 2 ✅

wrapper
None
func2
Function 2


In [10]:
'''
题目4：梯度裁剪装饰器
功能：在反向传播后对梯度进行裁剪
参数：
- max_norm: 梯度最大范数

要求：
1. 装饰器应该作用于模型的训练步骤函数
2. 在loss.backward()之后，optimizer.step()之前进行梯度裁剪
3. 使用torch.nn.utils.clip_grad_norm_进行裁剪
4. 打印裁剪前后的梯度范数
'''
def gradient_clip_decorator(max_norm=1.0):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return decorator


In [ ]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)
        self.weights = nn.Parameter(torch.randn(10, 5))

    @gradient_clip_decorator(max_norm=0.5)
    def training_step(self, x, y, optimizer):
        """训练步骤，包含前向传播、损失计算、反向传播"""
        predictions = x @ self.weights
        loss = ((predictions - y) ** 2).mean()
        loss.backward()

        # 装饰器应该在这里插入梯度裁剪逻辑
        # 然后执行optimizer.step()
        optimizer.step()
        optimizer.zero_grad()

        return loss

    @gradient_clip_decorator(max_norm=1.0)
    def training_step_with_nn(self, x, y, optimizer):
        """使用nn.Module参数的训练步骤"""
        predictions = self.linear(x)
        loss = nn.functional.mse_loss(predictions, y)
        loss.backward()

        # 装饰器应该在这里插入梯度裁剪逻辑
        optimizer.step()
        optimizer.zero_grad()

        return loss